<a href="https://colab.research.google.com/github/musa123-debug/task-feature-risk-signal-audit/blob/main/construction_risk_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
print("Upload construction_dataset.csv")
uploaded = files.upload()

Upload construction_dataset.csv


Saving construction_dataset.csv to construction_dataset.csv


In [ ]:
# =====================================================================
# FIGURE GENERATION — produces fig1.png ... fig8.png
# Run AFTER the main analysis cell (depends on its variables).
# =====================================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300,
                     'savefig.bbox': 'tight', 'font.size': 10})

MAJ_ACC, MAJ_F1 = 0.5077, 0.224
STRAT_ACC, STRAT_F1 = 0.3769, 0.326

# ---------------------------------------------------------- FIGURE 1
fig, ax = plt.subplots(figsize=(6,4))
counts = [660, 377, 263]
bars = ax.bar(['Low','Medium','High'], counts,
              color=['#4C72B0','#DD8452','#C44E52'])
for b,c in zip(bars,counts):
    ax.text(b.get_x()+b.get_width()/2, c+8, str(c), ha='center')
ax.set_ylabel('Number of Tasks'); ax.set_xlabel('Risk Level')
ax.set_title('Class Distribution (n = 1,300)')
ax.set_ylim(0, 720)
plt.savefig('fig1.png'); plt.close()

# ---------------------------------------------------------- FIGURE 2
fig, ax = plt.subplots(figsize=(7,4))
items = sorted(corr.items(), key=lambda kv: kv[1])
names = [k for k,_ in items]; vals = [v for _,v in items]
ax.barh(names, vals, color=['#C44E52' if v<0 else '#4C72B0' for v in vals])
ax.axvline(0, color='k', lw=0.8)
ax.set_xlim(-0.10, 0.10)
ax.set_xlabel('Pearson correlation coefficient (r)')
ax.set_title('Feature–Label Correlation  (max |r| = 0.0596)')
plt.savefig('fig2.png'); plt.close()

# ---------------------------------------------------------- FIGURE 3
cfg = [('LR\ndefault',      R['lr0']['acc']),
       ('LR\ntuned',        R['LR_t']['acc']),
       ('RF\ndefault',      R['rf0']['acc']),
       ('RF\nweighted',     R['rfw']['acc']),
       ('RF\ntuned',        R['RF_t']['acc']),
       ('RF\ntuned-F1',     R['RF_f1']['acc']),
       ('XGB\ndefault',     R['xgb0']['acc']),
       ('XGB\nweighted',    R['xgbw']['acc']),
       ('XGB\ntuned',       R['XGB_t']['acc']),
       ('XGB\ntuned-F1',    R['XGB_f1']['acc'])]
fig, ax = plt.subplots(figsize=(10,4.5))
bars = ax.bar([c[0] for c in cfg], [c[1] for c in cfg], color='#4C72B0')
for b,(_,v) in zip(bars,cfg):
    ax.text(b.get_x()+b.get_width()/2, v+0.008, f'{v*100:.1f}%',
            ha='center', fontsize=8)
ax.axhline(MAJ_ACC, color='k', ls='--', lw=1.2,
           label=f'Majority-class baseline ({MAJ_ACC*100:.2f}%)')
ax.axhline(STRAT_ACC, color='grey', ls='--', lw=1.2,
           label=f'Stratified random guesser ({STRAT_ACC*100:.2f}%)')
ax.set_ylabel('Test accuracy'); ax.set_ylim(0, 0.65)
ax.set_title('Test-Set Accuracy Across All Configurations (n = 260)')
ax.legend(fontsize=8)
plt.savefig('fig3.png'); plt.close()

# ---------------------------------------------------------- FIGURE 4
fig, ax = plt.subplots(figsize=(7,4))
g = sorted(gain.items(), key=lambda kv: kv[1])
ax.barh([k for k,_ in g], [v for _,v in g], color='#4C72B0')
ax.axvline(0.125, color='r', ls='--', lw=1,
           label='Uniform allocation (0.125)')
for i,(k,v) in enumerate(g):
    ax.text(v+0.002, i, f'{v:.4f}', va='center', fontsize=8)
ax.set_xlim(0, 0.19)
ax.set_xlabel('Gain-based importance')
ax.set_title('XGBoost Feature Importance (tuned)')
ax.legend(fontsize=8)
plt.savefig('fig4.png'); plt.close()

# ---------------------------------------------------------- FIGURE 5
cm = confusion_matrix(yte, R['RF_t']['pred'])
fig, ax = plt.subplots(figsize=(5.5,4.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES)
ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
ax.set_title('Confusion Matrix — Tuned Random Forest')
fig.colorbar(im)
plt.savefig('fig5.png'); plt.close()

# ---------------------------------------------------------- FIGURE 6
f1s = [('LR\ndefault',   R['lr0']['mf1']),  ('LR\ntuned',     R['LR_t']['mf1']),
       ('RF\ndefault',   R['rf0']['mf1']),  ('RF\nweighted',  R['rfw']['mf1']),
       ('RF\ntuned',     R['RF_t']['mf1']), ('RF\ntuned-F1',  R['RF_f1']['mf1']),
       ('XGB\ndefault',  R['xgb0']['mf1']), ('XGB\nweighted', R['xgbw']['mf1']),
       ('XGB\ntuned',    R['XGB_t']['mf1']),('XGB\ntuned-F1', R['XGB_f1']['mf1'])]
fig, ax = plt.subplots(figsize=(10,4.5))
bars = ax.bar([c[0] for c in f1s], [c[1] for c in f1s], color='#DD8452')
for b,(_,v) in zip(bars,f1s):
    ax.text(b.get_x()+b.get_width()/2, v+0.006, f'{v:.3f}',
            ha='center', fontsize=8)
ax.axhline(MAJ_F1, color='k', ls='--', lw=1.2,
           label=f'Majority-class baseline ({MAJ_F1:.3f})')
ax.axhline(STRAT_F1, color='grey', ls='--', lw=1.2,
           label=f'Stratified random guesser ({STRAT_F1:.3f})')
ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 0.45)
ax.set_title('Macro-F1 Across All Configurations')
ax.legend(fontsize=8)
plt.savefig('fig6.png'); plt.close()

# ---------------------------------------------------------- FIGURE 7
fig, ax = plt.subplots(figsize=(7,4.5))
ax.hist(null*100, bins=30, color='#8C8C8C', edgecolor='white')
ax.axvline(obs*100, color='r', lw=2,
           label=f'Observed (true labels): {obs*100:.2f}%')
ax.axvline(np.percentile(null,95)*100, color='k', ls='--', lw=1.2,
           label=f'Null 95th percentile: {np.percentile(null,95)*100:.2f}%')
ax.set_xlabel('Test accuracy (%)'); ax.set_ylabel('Frequency')
ax.set_title(f'Label Permutation Test — 1,000 shuffles  (p = {p_perm:.4f})')
ax.legend(fontsize=8)
plt.savefig('fig7.png'); plt.close()

# ---------------------------------------------------------- FIGURE 8
seps, margins = [], []
for sep in [0.02,0.04,0.06,0.08,0.10,0.20,0.30,0.50,0.75,1.00]:
    Xs, ys = make_classification(n_samples=1300, n_features=8, n_informative=5,
                                 n_redundant=0, n_classes=3,
                                 weights=[0.508,0.290,0.202],
                                 class_sep=sep, random_state=SEED)
    Xs = pd.DataFrame(Xs, columns=[f'f{i}' for i in range(8)])
    a_, b_, c_, d_ = train_test_split(Xs, ys, test_size=0.2,
                                      random_state=SEED, stratify=ys)
    m = RandomForestClassifier(**bp, random_state=SEED, n_jobs=-1).fit(a_, c_)
    acc_  = accuracy_score(d_, m.predict(b_))
    base_ = accuracy_score(d_, DummyClassifier(strategy='most_frequent')
                                 .fit(a_,c_).predict(b_))
    seps.append(sep); margins.append((acc_-base_)*100)

fig, ax = plt.subplots(figsize=(7.5,4.5))
ax.plot(seps, margins, 'o-', color='#4C72B0', label='Synthetic control')
ax.axhline(0.77, color='r', ls='--', lw=1.5,
           label='Real dataset (+0.77 pts)')
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('class_sep (synthetic signal strength)')
ax.set_ylabel('Accuracy margin over majority baseline (pts)')
ax.set_title('Positive Control — Diagnostic Fires at Every Separation Tested')
ax.legend(fontsize=8)
plt.savefig('fig8.png'); plt.close()

print("Wrote fig1.png ... fig8.png")

from google.colab import files
for i in range(1, 9):
    files.download(f'fig{i}.png')

NameError: name 'corr' is not defined

In [ ]:
# =====================================================================
# Task-Level Features Show No Detectable Risk Signal in a Public
# Construction Dataset: A Diagnostic Audit
# Full reproducible analysis. Regenerates every number in the paper.
# =====================================================================

from google.colab import files
print("Upload construction_dataset.csv")
uploaded = files.upload()

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.datasets import make_classification
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
from xgboost import XGBClassifier
import warnings; warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ------------------------------------------------------------ 1. DATA
df = pd.read_csv('construction_dataset.csv')
X  = df.drop(columns=['Task_ID', 'Risk_Level'])
le = LabelEncoder(); y = le.fit_transform(df['Risk_Level'])
CLASSES, FEATURES = list(le.classes_), list(X.columns)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                      random_state=SEED, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
N = len(yte)

print("\n" + "="*72); print("1. DATASET"); print("="*72)
print(f"Shape {df.shape} | Classes {CLASSES} | Train/Test {len(ytr)}/{N}")
vc = df['Risk_Level'].value_counts()
for c in ['Low','Medium','High']:
    print(f"  {c:7s} {vc[c]:4d} ({vc[c]/len(df)*100:.1f}%)")
print(f"Missing {df.isnull().sum().sum()} | Duplicates {df.duplicated().sum()}")
print(f"Task_Duration_Days {df.Task_Duration_Days.min()}-{df.Task_Duration_Days.max()}")
print(f"Material_Cost_USD ${df.Material_Cost_USD.min():,.0f}-${df.Material_Cost_USD.max():,.0f}")

# ---------------------------------------------------- 2. CORRELATIONS
print("\n" + "="*72); print("2. FEATURE-LABEL CORRELATION"); print("="*72)
corr = {f: np.corrcoef(X[f], y)[0,1] for f in FEATURES}
for f, v in sorted(corr.items(), key=lambda kv: -abs(kv[1])):
    print(f"  {f:26s} r = {v:+.4f}")
print(f"  Max |r| = {max(abs(v) for v in corr.values()):.4f}")

# --------------------------------------------------------- 3. HELPERS
def ev(name, model, weighted=False, quiet=False):
    if weighted:
        model.fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', y=ytr))
    else:
        model.fit(Xtr, ytr)
    p = model.predict(Xte)
    r = dict(pred=p,
             acc=accuracy_score(yte, p),
             mf1=f1_score(yte, p, average='macro', zero_division=0),
             mp=precision_score(yte, p, average='macro', zero_division=0),
             mr=recall_score(yte, p, average='macro', zero_division=0),
             hr=recall_score(yte, p, average=None, zero_division=0)[CLASSES.index('High')])
    if not quiet:
        print(f"\n--- {name} ---")
        print(f"  Accuracy {r['acc']*100:.2f}% ({(p==yte).sum()}/{N}) | "
              f"macro P/R/F1 {r['mp']:.3f}/{r['mr']:.3f}/{r['mf1']:.3f} | "
              f"High recall {r['hr']:.3f}")
        print("  Confusion (rows=true " + str(CLASSES) + "):")
        print("  " + str(confusion_matrix(yte, p)).replace("\n", "\n  "))
    return r

R = {}

# ------------------------------------------------- 4. INITIAL MODELS
print("\n" + "="*72); print("4. INITIAL MODELS (untuned, unweighted)"); print("="*72)
R['lr0']  = ev("Logistic Regression (default)",
               make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)))
R['rf0']  = ev("Random Forest (100 trees, depth 10)",
               RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1))
R['xgb0'] = ev("XGBoost (200 est, depth 5, lr 0.3)",
               XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.3,
                             random_state=SEED, verbosity=0))

# ------------------------------------------------- 5. CLASS-WEIGHTED
print("\n" + "="*72); print("5. CLASS-WEIGHTED MODELS (untuned)"); print("="*72)
R['rfw']  = ev("Random Forest (class-weighted)",
               RandomForestClassifier(n_estimators=100, max_depth=10,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1))
R['xgbw'] = ev("XGBoost (class-weighted)",
               XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.3,
                             random_state=SEED, verbosity=0), weighted=True)

# ------------------------------------------------------------- 6. CV
print("\n" + "="*72); print("6. 5-FOLD CROSS-VALIDATION"); print("="*72)
for nm, m in [
    ("LR (default)",           make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))),
    ("RF unweighted (100/10)", RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED, n_jobs=-1)),
    ("RF weighted (100/10)",   RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=SEED, n_jobs=-1)),
    ("XGB unweighted",         XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.3, random_state=SEED, verbosity=0)),
]:
    s = cross_val_score(m, X, y, cv=cv, scoring='accuracy')
    print(f"  {nm:26s} {s.mean()*100:.2f}% (± {s.std()*100:.2f}%)")

# ------------------------------------------ 7. GRIDSEARCH (accuracy)
print("\n" + "="*72); print("7. GRIDSEARCH (accuracy scoring)"); print("="*72)
GA = {}
GA['LR'] = GridSearchCV(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
                        {'logisticregression__C':[0.001,0.01,0.1,1,10,100]},
                        cv=cv, scoring='accuracy', n_jobs=-1).fit(Xtr, ytr)
GA['RF'] = GridSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=-1),
                        {'n_estimators':[50,100,200],'max_depth':[5,10,15],'min_samples_split':[2,5,10]},
                        cv=cv, scoring='accuracy', n_jobs=-1).fit(Xtr, ytr)
GA['XGB']= GridSearchCV(XGBClassifier(random_state=SEED, verbosity=0),
                        {'n_estimators':[100,200,300],'max_depth':[3,5,7],'learning_rate':[0.01,0.1,0.3]},
                        cv=cv, scoring='accuracy', n_jobs=-1).fit(Xtr, ytr)
for k in ['LR','RF','XGB']:
    print(f"  {k:4s} best={GA[k].best_params_} CV={GA[k].best_score_*100:.2f}%")
for k in ['LR','RF','XGB']:
    R[f'{k}_t'] = ev(f"{k} tuned", GA[k].best_estimator_)

# ------------------------------------------ 8. GRIDSEARCH (macro-F1)
print("\n" + "="*72); print("8. GRIDSEARCH (macro-F1 scoring)"); print("="*72)
GF = {}
GF['RF'] = GridSearchCV(RandomForestClassifier(random_state=SEED, n_jobs=-1),
                        {'n_estimators':[50,100,200],'max_depth':[5,10,15],'min_samples_split':[2,5,10]},
                        cv=cv, scoring='f1_macro', n_jobs=-1).fit(Xtr, ytr)
GF['XGB']= GridSearchCV(XGBClassifier(random_state=SEED, verbosity=0),
                        {'n_estimators':[100,200,300],'max_depth':[3,5,7],'learning_rate':[0.01,0.1,0.3]},
                        cv=cv, scoring='f1_macro', n_jobs=-1).fit(Xtr, ytr)
for k in ['RF','XGB']:
    print(f"  {k:4s} best={GF[k].best_params_} CV macro-F1={GF[k].best_score_:.3f}")
    R[f'{k}_f1'] = ev(f"{k} F1-optimized", GF[k].best_estimator_)

# --------------------------------------------- 9. FEATURE IMPORTANCE
print("\n" + "="*72); print("9. FEATURE IMPORTANCE"); print("="*72)
gain = dict(zip(FEATURES, GA['XGB'].best_estimator_.feature_importances_))
print("XGBoost gain-based:")
for f, v in sorted(gain.items(), key=lambda kv: -kv[1]):
    print(f"  {f:26s} {v:.4f}")
print(f"  Range {max(gain.values()):.4f} to {min(gain.values()):.4f}")

pi = permutation_importance(GA['RF'].best_estimator_, Xte, yte,
                            n_repeats=30, random_state=SEED, n_jobs=-1)
print("\nPermutation importance (tuned RF, 30 repeats):")
for i in np.argsort(-pi.importances_mean):
    print(f"  {FEATURES[i]:26s} {pi.importances_mean[i]:+.4f} ± {pi.importances_std[i]:.4f}")

# -------------------------------------------- 10. FEATURE ENGINEERING
print("\n" + "="*72); print("10. FEATURE ENGINEERING"); print("="*72)
Xe = X.copy()
Xe['Duration_x_Labor']   = X.Task_Duration_Days * X.Labor_Required
Xe['Cost_x_Resource']    = X.Material_Cost_USD  * X.Resource_Constraint_Score
Xe['Cost_x_Site']        = X.Material_Cost_USD  * X.Site_Constraint_Score
Xe['Duration_sq']        = X.Task_Duration_Days ** 2
Xe['Constraint_Sum']     = X.Resource_Constraint_Score + X.Site_Constraint_Score
Xe['Constraint_Product'] = X.Resource_Constraint_Score * X.Site_Constraint_Score
for k in ['RF','XGB']:
    b = cross_val_score(GA[k].best_estimator_, X,  y, cv=cv, scoring='accuracy').mean()
    e = cross_val_score(GA[k].best_estimator_, Xe, y, cv=cv, scoring='accuracy').mean()
    print(f"  {k:4s} baseline CV {b*100:.2f}% -> engineered CV {e*100:.2f}% ({(e-b)*100:+.2f} pts)")

# ------------------------------------------------- 11. ERROR ANALYSIS
print("\n" + "="*72); print("11. ERROR ANALYSIS (class-weighted RF)"); print("="*72)
ok = (R['rfw']['pred'] == yte); Xr = Xte.reset_index(drop=True)
print(f"  Correct {ok.sum()} | Incorrect {(~ok).sum()}\n")
print(f"  {'Feature':26s} {'Correct':>11s} {'Incorrect':>11s} {'Diff':>10s}")
for f in FEATURES:
    a, b = Xr[f][ok].mean(), Xr[f][~ok].mean()
    print(f"  {f:26s} {a:11.2f} {b:11.2f} {b-a:+10.2f}")

# ------------------------------------- 12. BASELINES & SIGNIFICANCE
print("\n" + "="*72); print("12. BASELINES & SIGNIFICANCE"); print("="*72)
print("Reference levels:")
for s, lab in [('most_frequent','Majority-class baseline'),
               ('stratified','Stratified random guesser'),
               ('uniform','Uniform random guesser')]:
    d = DummyClassifier(strategy=s, random_state=SEED).fit(Xtr, ytr).predict(Xte)
    print(f"  {lab:26s} acc {accuracy_score(yte,d)*100:5.2f}%  "
          f"macro-F1 {f1_score(yte,d,average='macro',zero_division=0):.3f}")

maj = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr).predict(Xte)
mj  = (maj == yte)
print(f"\n{'Model':14s} {'Acc':>8s} {'95% Wilson CI':>20s} {'McNemar p':>11s}")
for k, lab in [('LR_t','LR tuned'), ('RF_t','RF tuned'), ('XGB_t','XGB tuned')]:
    mc = (R[k]['pred'] == yte); c = int(mc.sum())
    lo, hi = proportion_confint(c, N, alpha=0.05, method='wilson')
    tbl = [[int((mc&mj).sum()), int((mc&~mj).sum())],
           [int((~mc&mj).sum()), int((~mc&~mj).sum())]]
    print(f"{lab:14s} {c/N*100:7.2f}%  [{lo*100:5.1f}%, {hi*100:5.1f}%] "
          f"{mcnemar(tbl, exact=True).pvalue:11.4f}")
c = int(mj.sum()); lo, hi = proportion_confint(c, N, alpha=0.05, method='wilson')
print(f"{'Majority':14s} {c/N*100:7.2f}%  [{lo*100:5.1f}%, {hi*100:5.1f}%] {'—':>11s}")

# ---------------------------------------------- 13. PERMUTATION TEST
print("\n" + "="*72); print("13. LABEL PERMUTATION TEST (1000 shuffles)"); print("="*72)
bp  = GA['RF'].best_params_
obs = R['RF_t']['acc']
rng, null = np.random.default_rng(SEED), []
for i in range(1000):
    m = RandomForestClassifier(**bp, random_state=SEED, n_jobs=-1)
    m.fit(Xtr, rng.permutation(ytr))
    null.append(accuracy_score(yte, m.predict(Xte)))
    if (i+1) % 250 == 0: print(f"  ...{i+1}/1000")
null = np.array(null)
p_perm = (np.sum(null >= obs) + 1) / (len(null) + 1)
print(f"\n  Observed {obs*100:.2f}% | Null mean {null.mean()*100:.2f}% "
      f"(SD {null.std()*100:.2f}%) | 95th pct {np.percentile(null,95)*100:.2f}%")
print(f"  Permutation p = {p_perm:.4f}")

# -------------------------- 14. POSITIVE CONTROL & SENSITIVITY SWEEP
print("\n" + "="*72); print("14. POSITIVE CONTROL — SENSITIVITY SWEEP"); print("="*72)
print("Does the diagnostic sequence detect signal where signal exists?\n")
print(f"{'class_sep':>10} {'max|r|':>8} {'RF acc':>9} {'margin':>9} "
      f"{'macro-F1':>9} {'High rec':>9} {'perm p':>8}")
print("-"*70)
for sep in [0.02,0.04,0.06,0.08,0.10,0.20,0.30,0.50,0.75,1.00]:
    Xs, ys = make_classification(n_samples=1300, n_features=8, n_informative=5,
                                 n_redundant=0, n_classes=3,
                                 weights=[0.508,0.290,0.202],
                                 class_sep=sep, random_state=SEED)
    Xs = pd.DataFrame(Xs, columns=[f'f{i}' for i in range(8)])
    a, b_, c_, d_ = train_test_split(Xs, ys, test_size=0.2, random_state=SEED, stratify=ys)
    mr_ = max(abs(np.corrcoef(Xs[c][:], ys)[0,1]) for c in Xs.columns)
    m = RandomForestClassifier(**bp, random_state=SEED, n_jobs=-1).fit(a, c_)
    pr = m.predict(b_)
    acc_ = accuracy_score(d_, pr)
    base_= accuracy_score(d_, DummyClassifier(strategy='most_frequent').fit(a,c_).predict(b_))
    rg, nl = np.random.default_rng(SEED), []
    for _ in range(200):
        mm = RandomForestClassifier(**bp, random_state=SEED, n_jobs=-1)
        mm.fit(a, rg.permutation(c_))
        nl.append(accuracy_score(d_, mm.predict(b_)))
    nl = np.array(nl); pv = (np.sum(nl >= acc_)+1)/(len(nl)+1)
    print(f"{sep:>10.2f} {mr_:>8.4f} {acc_*100:>8.2f}% {(acc_-base_)*100:>+8.2f} "
          f"{f1_score(d_,pr,average='macro'):>9.3f} "
          f"{recall_score(d_,pr,average=None,zero_division=0)[2]:>9.3f} {pv:>8.4f}")
print("-"*70)
print(f"{'REAL DATA':>10} {0.0596:>8.4f} {obs*100:>8.2f}% {+0.77:>+8.2f} "
      f"{R['RF_t']['mf1']:>9.3f} {R['RF_t']['hr']:>9.3f} {p_perm:>8.4f}")

print("\n" + "="*72); print("ANALYSIS COMPLETE"); print("="*72)

Upload construction_dataset.csv


Saving construction_dataset.csv to construction_dataset (1).csv

1. DATASET
Shape (1300, 10) | Classes ['High', 'Low', 'Medium'] | Train/Test 1040/260
  Low      660 (50.8%)
  Medium   377 (29.0%)
  High     263 (20.2%)
Missing 0 | Duplicates 0
Task_Duration_Days 1-89
Material_Cost_USD $1,003-$99,956

2. FEATURE-LABEL CORRELATION
  Start_Constraint           r = -0.0596
  Resource_Constraint_Score  r = +0.0372
  Dependency_Count           r = +0.0336
  Site_Constraint_Score      r = +0.0243
  Labor_Required             r = -0.0174
  Material_Cost_USD          r = +0.0170
  Task_Duration_Days         r = -0.0101
  Equipment_Units            r = -0.0054
  Max |r| = 0.0596

4. INITIAL MODELS (untuned, unweighted)

--- Logistic Regression (default) ---
  Accuracy 50.77% (132/260) | macro P/R/F1 0.169/0.333/0.224 | High recall 0.000
  Confusion (rows=true ['High', 'Low', 'Medium']):
  [[  0  53   0]
   [  0 132   0]
   [  0  75   0]]

--- Random Forest (100 trees, depth 10) ---
  Accuracy 5

In [ ]:
# =====================================================================
# FIGURE GENERATION — produces fig1.png ... fig8.png
# =====================================================================
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300,
                     'savefig.bbox': 'tight', 'font.size': 10})

MAJ_ACC, MAJ_F1 = 0.5077, 0.224
STRAT_ACC, STRAT_F1 = 0.3769, 0.326

# ---------------------------------------------------------- FIGURE 1
fig, ax = plt.subplots(figsize=(6,4))
counts = [660, 377, 263]
bars = ax.bar(['Low','Medium','High'], counts,
              color=['#4C72B0','#DD8452','#C44E52'])
for b,c in zip(bars,counts):
    ax.text(b.get_x()+b.get_width()/2, c+8, str(c), ha='center')
ax.set_ylabel('Number of Tasks'); ax.set_xlabel('Risk Level')
ax.set_title('Class Distribution (n = 1,300)')
ax.set_ylim(0, 720)
plt.savefig('fig1.png'); plt.close()

# ---------------------------------------------------------- FIGURE 2
fig, ax = plt.subplots(figsize=(7,4))
items = sorted(corr.items(), key=lambda kv: kv[1])
names = [k for k,_ in items]; vals = [v for _,v in items]
ax.barh(names, vals, color=['#C44E52' if v<0 else '#4C72B0' for v in vals])
ax.axvline(0, color='k', lw=0.8)
ax.set_xlim(-0.10, 0.10)
ax.set_xlabel('Pearson correlation coefficient (r)')
ax.set_title('Feature-Label Correlation  (max |r| = 0.0596)')
plt.savefig('fig2.png'); plt.close()

# ---------------------------------------------------------- FIGURE 3
cfg = [('LR\ndefault',   R['lr0']['acc']),  ('LR\ntuned',     R['LR_t']['acc']),
       ('RF\ndefault',   R['rf0']['acc']),  ('RF\nweighted',  R['rfw']['acc']),
       ('RF\ntuned',     R['RF_t']['acc']), ('RF\ntuned-F1',  R['RF_f1']['acc']),
       ('XGB\ndefault',  R['xgb0']['acc']), ('XGB\nweighted', R['xgbw']['acc']),
       ('XGB\ntuned',    R['XGB_t']['acc']),('XGB\ntuned-F1', R['XGB_f1']['acc'])]
fig, ax = plt.subplots(figsize=(10,4.5))
bars = ax.bar([c[0] for c in cfg], [c[1] for c in cfg], color='#4C72B0')
for b,(_,v) in zip(bars,cfg):
    ax.text(b.get_x()+b.get_width()/2, v+0.008, f'{v*100:.1f}%',
            ha='center', fontsize=8)
ax.axhline(MAJ_ACC, color='k', ls='--', lw=1.2,
           label=f'Majority-class baseline ({MAJ_ACC*100:.2f}%)')
ax.axhline(STRAT_ACC, color='grey', ls='--', lw=1.2,
           label=f'Stratified random guesser ({STRAT_ACC*100:.2f}%)')
ax.set_ylabel('Test accuracy'); ax.set_ylim(0, 0.65)
ax.set_title('Test-Set Accuracy Across All Configurations (n = 260)')
ax.legend(fontsize=8)
plt.savefig('fig3.png'); plt.close()

# ---------------------------------------------------------- FIGURE 4
fig, ax = plt.subplots(figsize=(7,4))
g = sorted(gain.items(), key=lambda kv: kv[1])
ax.barh([k for k,_ in g], [v for _,v in g], color='#4C72B0')
ax.axvline(0.125, color='r', ls='--', lw=1, label='Uniform allocation (0.125)')
for i,(k,v) in enumerate(g):
    ax.text(v+0.002, i, f'{v:.4f}', va='center', fontsize=8)
ax.set_xlim(0, 0.19)
ax.set_xlabel('Gain-based importance')
ax.set_title('XGBoost Feature Importance (tuned)')
ax.legend(fontsize=8)
plt.savefig('fig4.png'); plt.close()

# ---------------------------------------------------------- FIGURE 5
cm = confusion_matrix(yte, R['RF_t']['pred'])
fig, ax = plt.subplots(figsize=(5.5,4.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES)
ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
ax.set_title('Confusion Matrix - Tuned Random Forest')
fig.colorbar(im)
plt.savefig('fig5.png'); plt.close()

# ---------------------------------------------------------- FIGURE 6
f1s = [('LR\ndefault',   R['lr0']['mf1']),  ('LR\ntuned',     R['LR_t']['mf1']),
       ('RF\ndefault',   R['rf0']['mf1']),  ('RF\nweighted',  R['rfw']['mf1']),
       ('RF\ntuned',     R['RF_t']['mf1']), ('RF\ntuned-F1',  R['RF_f1']['mf1']),
       ('XGB\ndefault',  R['xgb0']['mf1']), ('XGB\nweighted', R['xgbw']['mf1']),
       ('XGB\ntuned',    R['XGB_t']['mf1']),('XGB\ntuned-F1', R['XGB_f1']['mf1'])]
fig, ax = plt.subplots(figsize=(10,4.5))
bars = ax.bar([c[0] for c in f1s], [c[1] for c in f1s], color='#DD8452')
for b,(_,v) in zip(bars,f1s):
    ax.text(b.get_x()+b.get_width()/2, v+0.006, f'{v:.3f}',
            ha='center', fontsize=8)
ax.axhline(MAJ_F1, color='k', ls='--', lw=1.2,
           label=f'Majority-class baseline ({MAJ_F1:.3f})')
ax.axhline(STRAT_F1, color='grey', ls='--', lw=1.2,
           label=f'Stratified random guesser ({STRAT_F1:.3f})')
ax.set_ylabel('Macro-F1'); ax.set_ylim(0, 0.45)
ax.set_title('Macro-F1 Across All Configurations')
ax.legend(fontsize=8)
plt.savefig('fig6.png'); plt.close()

# ---------------------------------------------------------- FIGURE 7
fig, ax = plt.subplots(figsize=(7,4.5))
ax.hist(null*100, bins=30, color='#8C8C8C', edgecolor='white')
ax.axvline(obs*100, color='r', lw=2,
           label=f'Observed (true labels): {obs*100:.2f}%')
ax.axvline(np.percentile(null,95)*100, color='k', ls='--', lw=1.2,
           label=f'Null 95th percentile: {np.percentile(null,95)*100:.2f}%')
ax.set_xlabel('Test accuracy (%)'); ax.set_ylabel('Frequency')
ax.set_title(f'Label Permutation Test - 1,000 shuffles  (p = {p_perm:.4f})')
ax.legend(fontsize=8)
plt.savefig('fig7.png'); plt.close()

# ---------------------------------------------------------- FIGURE 8
seps, margins = [], []
for sep in [0.02,0.04,0.06,0.08,0.10,0.20,0.30,0.50,0.75,1.00]:
    Xs, ys = make_classification(n_samples=1300, n_features=8, n_informative=5,
                                 n_redundant=0, n_classes=3,
                                 weights=[0.508,0.290,0.202],
                                 class_sep=sep, random_state=SEED)
    Xs = pd.DataFrame(Xs, columns=[f'f{i}' for i in range(8)])
    a_, b_, c_, d_ = train_test_split(Xs, ys, test_size=0.2,
                                      random_state=SEED, stratify=ys)
    m = RandomForestClassifier(**bp, random_state=SEED, n_jobs=-1).fit(a_, c_)
    acc_  = accuracy_score(d_, m.predict(b_))
    base_ = accuracy_score(d_, DummyClassifier(strategy='most_frequent')
                                 .fit(a_,c_).predict(b_))
    seps.append(sep); margins.append((acc_-base_)*100)

fig, ax = plt.subplots(figsize=(7.5,4.5))
ax.plot(seps, margins, 'o-', color='#4C72B0', label='Synthetic control')
ax.axhline(0.77, color='r', ls='--', lw=1.5, label='Real dataset (+0.77 pts)')
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('class_sep (synthetic signal strength)')
ax.set_ylabel('Accuracy margin over majority baseline (pts)')
ax.set_title('Positive Control - Diagnostic Fires at Every Separation Tested')
ax.legend(fontsize=8)
plt.savefig('fig8.png'); plt.close()

print("Wrote fig1.png ... fig8.png")

Wrote fig1.png ... fig8.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>